# Redrob Dataset EDA

Exploratory analysis notebook for the Redrob candidate dataset. This notebook intentionally avoids model training.

In [ ]:
from pathlib import Path
import json
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

ROOT = Path.cwd()
DATA_DIR = ROOT / '[PUB] India_runs_data_and_ai_challenge' / '[PUB] India_runs_data_and_ai_challenge' / 'India_runs_data_and_ai_challenge'
CANDIDATES_PATH = DATA_DIR / 'candidates.jsonl'
SCHEMA_PATH = DATA_DIR / 'candidate_schema.json'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'
print(CANDIDATES_PATH)

In [ ]:
records = []
with CANDIDATES_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

len(records), records[0].keys()

## Flatten Candidate-Level Fields

This keeps one row per candidate and derives simple aggregate fields from nested collections.

In [ ]:
def get_path(obj, path, default=None):
    cur = obj
    for part in path.split('.'):
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return default
    return cur

rows = []
for r in records:
    sig = r.get('redrob_signals', {})
    skills = r.get('skills', []) or []
    career = r.get('career_history', []) or []
    education = r.get('education', []) or []
    certs = r.get('certifications', []) or []
    languages = r.get('languages', []) or []
    assessments = sig.get('skill_assessment_scores', {}) or {}
    rows.append({
        'candidate_id': r.get('candidate_id'),
        'headline': get_path(r, 'profile.headline'),
        'summary': get_path(r, 'profile.summary'),
        'location': get_path(r, 'profile.location'),
        'country': get_path(r, 'profile.country'),
        'years_of_experience': get_path(r, 'profile.years_of_experience'),
        'current_title': get_path(r, 'profile.current_title'),
        'current_company': get_path(r, 'profile.current_company'),
        'current_company_size': get_path(r, 'profile.current_company_size'),
        'current_industry': get_path(r, 'profile.current_industry'),
        'career_count': len(career),
        'education_count': len(education),
        'skills_count': len(skills),
        'certifications_count': len(certs),
        'languages_count': len(languages),
        'total_skill_endorsements': sum(s.get('endorsements', 0) for s in skills),
        'max_skill_duration_months': max([s.get('duration_months', 0) for s in skills], default=0),
        'assessment_count': len(assessments),
        'assessment_mean_score': np.mean(list(assessments.values())) if assessments else np.nan,
        'salary_min_lpa': get_path(r, 'redrob_signals.expected_salary_range_inr_lpa.min'),
        'salary_max_lpa': get_path(r, 'redrob_signals.expected_salary_range_inr_lpa.max'),
        **{k: v for k, v in sig.items() if k not in ['skill_assessment_scores', 'expected_salary_range_inr_lpa']}
    })

df = pd.DataFrame(rows)
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['last_active_date'] = pd.to_datetime(df['last_active_date'])
df['days_since_signup_to_active'] = (df['last_active_date'] - df['signup_date']).dt.days
df['salary_mid_lpa'] = (df['salary_min_lpa'] + df['salary_max_lpa']) / 2
df['salary_spread_lpa'] = df['salary_max_lpa'] - df['salary_min_lpa']
df.head()

In [ ]:
df.shape, df.dtypes.sort_index()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(30)

## Target Columns

The input dataset has no supervised target labels. The challenge output is defined by the sample submission.

In [ ]:
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
sample_submission.head(), sample_submission.columns.tolist()

## Numerical Feature Overview

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
df[numeric_cols].describe().T

In [ ]:
cols = ['years_of_experience', 'profile_completeness_score', 'recruiter_response_rate', 'avg_response_time_hours', 'connection_count', 'notice_period_days', 'salary_mid_lpa']
axes = df[cols].hist(figsize=(14, 9), bins=40)
plt.tight_layout()

## Categorical and Text Feature Overview

In [ ]:
categorical_cols = ['country', 'current_title', 'current_company_size', 'current_industry', 'preferred_work_mode', 'open_to_work_flag', 'willing_to_relocate', 'verified_email', 'verified_phone', 'linkedin_connected']
for col in categorical_cols:
    display(df[col].value_counts(dropna=False).head(20).to_frame('count'))

In [ ]:
text_cols = ['headline', 'summary', 'location', 'current_title', 'current_company', 'current_industry']
text_profile = pd.DataFrame({
    'column': text_cols,
    'non_null': [df[c].notna().sum() for c in text_cols],
    'unique': [df[c].nunique(dropna=True) for c in text_cols],
    'mean_length': [df[c].dropna().astype(str).str.len().mean() for c in text_cols],
})
text_profile

## Nested Collections

In [ ]:
def explode_items(records, field):
    out = []
    for r in records:
        for item in r.get(field, []) or []:
            row = {'candidate_id': r['candidate_id']}
            row.update(item)
            out.append(row)
    return pd.DataFrame(out)

skills_df = explode_items(records, 'skills')
career_df = explode_items(records, 'career_history')
education_df = explode_items(records, 'education')
certs_df = explode_items(records, 'certifications')
languages_df = explode_items(records, 'languages')

{
    'skills': skills_df.shape,
    'career': career_df.shape,
    'education': education_df.shape,
    'certifications': certs_df.shape,
    'languages': languages_df.shape,
}

In [ ]:
display(skills_df['name'].value_counts().head(30).to_frame('skill_count'))
display(skills_df['proficiency'].value_counts().to_frame('proficiency_count'))
display(career_df['title'].value_counts().head(30).to_frame('career_title_count'))
display(education_df['tier'].value_counts().to_frame('education_tier_count'))

## Behavioral Signals

In [ ]:
behavior_cols = [
    'profile_completeness_score', 'open_to_work_flag', 'profile_views_received_30d',
    'applications_submitted_30d', 'recruiter_response_rate', 'avg_response_time_hours',
    'connection_count', 'endorsements_received', 'notice_period_days', 'preferred_work_mode',
    'willing_to_relocate', 'github_activity_score', 'search_appearance_30d',
    'saved_by_recruiters_30d', 'interview_completion_rate', 'offer_acceptance_rate',
    'verified_email', 'verified_phone', 'linkedin_connected'
]
df[behavior_cols].describe(include='all').T

In [ ]:
corr_cols = [c for c in behavior_cols if c in df.select_dtypes(include='number').columns]
corr = df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)), corr.index)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

## Data Quality Checks

In [ ]:
quality = {
    'duplicate_candidate_ids': df['candidate_id'].duplicated().sum(),
    'salary_min_gt_max': (df['salary_min_lpa'] > df['salary_max_lpa']).sum(),
    'last_active_before_signup': (df['last_active_date'] < df['signup_date']).sum(),
    'github_unlinked_sentinel': (df['github_activity_score'] == -1).sum(),
    'offer_no_history_sentinel': (df['offer_acceptance_rate'] == -1).sum(),
}
quality

## No Modeling Yet

Next steps should stay in EDA/feature-design territory: validate quality issues, decide representation for nested text and skills, and define a ranking rubric against the job description before building any model.